## Paso 0 — Cargamos las 18 tablas ya tipadas

Mismo patrón que los notebooks anteriores.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")


Tablas cargadas: 18


## Paso 1 — ¿SAP muestra el mismo salto de precio que encontró el EDA-1? (§8 del plan)

El EDA-1 encontró un salto real de ~17% en el flete pagado en marzo de
2026, conectado al alza del salario mínimo. Aquí miramos si el propio
catálogo de tarifas SAP (independiente de ese otro análisis) muestra un
salto similar por esas fechas — sería una confirmación cruzada entre los
dos EDA, no solo una coincidencia narrada.


In [2]:
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

tarifa_plaza["vigencia_mes"] = tarifa_plaza["vigencia"].dt.to_period("M")
mediana_por_mes = tarifa_plaza.groupby("vigencia_mes")["valor"].median()
print(mediana_por_mes.to_string())


vigencia_mes
2024-01    1900000.0
2024-03    7000000.0
2024-04    4845000.0
2024-05    3500000.0
2024-06    1950000.0
2024-07    3400000.0
2024-08    2400000.0
2024-09    3233264.0
2024-10    3400000.0
2024-11    3200000.0
2024-12    3600000.0
2025-01    4372000.0
2025-02    3450000.0
2025-03    3500000.0
2025-04    4700000.0
2025-05    2000000.0
2025-06    4500000.0
2025-07    3121000.0
2025-08    3000000.0
2025-09    3500000.0
2025-10    3189500.0
2025-11    3100000.0
2025-12    2389000.0
2026-01    3808000.0
2026-02    3400000.0
2026-03    3500000.0
2026-04    3900000.0
2026-05    3300000.0
2026-06    3000000.0
2026-07    3498000.0
2026-08    2980000.0
2026-09    3800000.0
2026-10    2300000.0
2026-11    1100000.0
2026-12    2100000.0
3999-12    4000589.0
Freq: M


## Paso 2 — Tendencia de VALOR en el tiempo, por línea de negocio (historico-sap)

Corrección de método: usamos `historico-sap` (versiones históricas
reales de cada tarifa), no `tarifa-plaza` (una sola foto actual) — y
separamos por línea de negocio para no mezclar categorías de precio muy
distinto en un solo número por mes.


In [3]:
historico_sap = tablas["historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy"]

historico_sap["vigencia_mes"] = historico_sap["vigencia"].dt.to_period("M")

tendencia_por_linea = (
    historico_sap.groupby(["vigencia_mes", "lineaNegocioDescripcion"])["valor"]
    .median()
    .unstack(level="lineaNegocioDescripcion")
)
print(tendencia_por_linea.to_string())


lineaNegocioDescripcion  Oferta de Expedición Devolución Contenedor  Oferta de Expedición Interior a Puerto  Oferta de Expedición Licitaciones  Oferta de Expedición Nacional  Oferta de Expedición Puerto a interior
vigencia_mes                                                                                                                                                                                                         
2024-01                                                         NaN                                     NaN                                NaN                      1700000.0                               1700000.0
2024-03                                                         NaN                               1100000.0                                NaN                      1100000.0                                     NaN
2024-05                                                   1500000.0                               3500000.0                                NaN  

## Paso 3 — Normalización fina de ciudades con rapidfuzz (§9 del plan)

En vez de exigir coincidencia exacta de texto (lo que hicimos en el
notebook 01), usamos coincidencia difusa: para cada ciudad del catálogo
propio, buscamos el nombre más parecido dentro del vocabulario de zonas
SAP, con un puntaje de qué tan parecido es. Mismo enfoque que usó el
EDA-1 para su normalización de ciudades.


In [4]:
import re
import unicodedata
from rapidfuzz import process, fuzz

def limpiar_nombre_ciudad(texto):
    if pd.isna(texto):
        return None
    sin_departamento = re.sub(r"\s*\([^)]*\)\s*$", "", texto).strip()
    sin_tildes = unicodedata.normalize("NFKD", sin_departamento).encode("ascii", "ignore").decode("ascii")
    return sin_tildes.upper()

catalogo_rutas = tablas["catalogo_rutas"]
rutas_validas = catalogo_rutas[~catalogo_rutas["origen"].isin(["222", "b", "tr"])].copy()
rutas_validas["origen_limpio"] = rutas_validas["origen"].apply(limpiar_nombre_ciudad)
rutas_validas["destino_limpio"] = rutas_validas["destino"].apply(limpiar_nombre_ciudad)

valores_sap_codigos = (
    set(tarifa_plaza["origen"].dropna().unique())
    | set(tarifa_plaza["destino"].dropna().unique())
    | set(historico_sap["origen"].dropna().unique())
    | set(historico_sap["destino"].dropna().unique())
)
# sorted() para que el orden sea determinista: 14 ciudades tienen tanto ZC_ como ZTC_
# (ej. ZC_BOGOTA y ZTC_BOGOTA existen ambos) — sin orden fijo, cual gana variaria entre corridas
vocabulario_sap = {re.sub(r"^ZT?C_", "", codigo).replace("_", " "): codigo for codigo in sorted(valores_sap_codigos)}
nombres_sap = list(vocabulario_sap.keys())

ciudades_propias = sorted(set(rutas_validas["origen_limpio"]) | set(rutas_validas["destino_limpio"]))

resultados_fuzzy = []
for ciudad in ciudades_propias:
    mejor = process.extractOne(ciudad, nombres_sap, scorer=fuzz.WRatio)
    resultados_fuzzy.append({
        "ciudad_propia": ciudad,
        "mejor_match_sap": mejor[0],
        "score": mejor[1],
        "codigo_sap": vocabulario_sap[mejor[0]],
    })

df_fuzzy = pd.DataFrame(resultados_fuzzy).sort_values("score", ascending=False)
print(df_fuzzy.to_string())


          ciudad_propia mejor_match_sap       score         codigo_sap
0          BARRANQUILLA    BARRANQUILLA  100.000000   ZTC_BARRANQUILLA
1                BOGOTA          BOGOTA  100.000000         ZTC_BOGOTA
3           BUCARAMANGA     BUCARAMANGA  100.000000    ZTC_BUCARAMANGA
4          BUENAVENTURA    BUENAVENTURA  100.000000    ZC_BUENAVENTURA
6             CARTAGENA       CARTAGENA  100.000000      ZTC_CARTAGENA
5                  CALI            CALI  100.000000           ZTC_CALI
9                CUCUTA          CUCUTA  100.000000         ZTC_CUCUTA
8            COPACABANA      COPACABANA  100.000000      ZC_COPACABANA
23                YUMBO           YUMBO  100.000000           ZC_YUMBO
20          SANTA MARTA     SANTA MARTA  100.000000     ZC_SANTA MARTA
10         DOSQUEBRADAS    DOSQUEBRADAS  100.000000    ZC_DOSQUEBRADAS
11                FUNZA           FUNZA  100.000000           ZC_FUNZA
12               GUARNE          GUARNE  100.000000          ZC_GUARNE
14    

## Paso 3b — Verificación con el catálogo real de zonas (tblSAPDANE + tblZonasTarificacionRelacion)

El Paso 3 (y la síntesis original de este notebook) declaraba que el
91,7% de rapidfuzz era "una aproximación nuestra, no el mecanismo
real", porque `tblsapdane` y `tblZonasTarificacionRelacion` no estaban
en ningún extracto — la recomendación pendiente era pedirlas
explícitamente. Ya se consiguieron ambas
(`analitica/00-fuentes/fuentes-corporativas/`). Aquí se reproduce el
mecanismo real (no una aproximación): para cada ciudad, (1) revisar si
ya es una zona directa en `tarifa-plaza`/`historico-sap`, y si no, (2)
buscarla como municipio DANE real y resolver su zona de tarificación vía
`tblZonasTarificacionRelacion` — igual en espíritu a los 2 niveles que
usa `TarifarioSapAdapter.BuscarConZonasAsync` en el backend.

In [5]:
RUTA_CORP = Path("../../00-fuentes/fuentes-corporativas")
dane = pd.read_csv(RUTA_CORP / "tblSAPDANE_202609142051.csv")
zonas_relacion = pd.read_csv(RUTA_CORP / "tblZonasTarificacionRelacion_202609142051.csv")


def quitar_parentesis(texto):
    return re.sub(r"\s*\([^)]*\)\s*$", "", texto).strip()


def normalizar_simple(texto):
    sin_tildes = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", sin_tildes).strip().upper()


dane["municipio_norm"] = dane["NOMBREMUNICIPIO"].apply(normalizar_simple)
dane_municipios = set(dane["municipio_norm"].unique())

zonas_relacion["zc_bare"] = zonas_relacion["IDZONATRANSPORTE"].apply(
    lambda z: normalizar_simple(quitar_parentesis(re.sub(r"^ZC_", "", z)))
)
mapa_zc_a_ztc = dict(zip(zonas_relacion["zc_bare"], zonas_relacion["IDZONATARIFICACION"]))


def resolver_zona_real(ciudad_limpia):
    # Tier 1: la ciudad ya es una zona directa en tarifa-plaza/historico-sap
    if ciudad_limpia in vocabulario_sap:
        return vocabulario_sap[ciudad_limpia], "tier1: zona directa en tarifa-plaza"

    # Tier 2: es un municipio DANE real (exacto o por contains, como CodigosDaneAsync)
    municipio = None
    if ciudad_limpia in dane_municipios:
        municipio = ciudad_limpia
    else:
        candidatos = [m for m in dane_municipios if len(m) > 3 and (m in ciudad_limpia or ciudad_limpia in m)]
        if candidatos:
            municipio = min(candidatos, key=len)
    if municipio is None:
        return None, "no es municipio DANE reconocido"

    # Tier 2: resolver su zona vía la tabla de relación (exacto o por contains,
    # porque el nombre oficial DANE a veces difiere del usado en la relación,
    # ej. DANE dice "CARTAGENA DE INDIAS", la relación dice "CARTAGENA")
    if municipio in mapa_zc_a_ztc:
        return mapa_zc_a_ztc[municipio], f"tier2: '{municipio}' vía ZonasTarificacionRelacion"
    candidatos_zc = [k for k in mapa_zc_a_ztc if len(k) > 3 and (k in municipio or municipio in k)]
    if candidatos_zc:
        mejor = min(candidatos_zc, key=len)
        return mapa_zc_a_ztc[mejor], f"tier2: '{municipio}' ~ '{mejor}' vía ZonasTarificacionRelacion"

    return None, f"municipio DANE '{municipio}' válido, sin zona de tarificación asociada"


resultados_reales = [
    {"ciudad_propia": c, **dict(zip(["zona_real", "motivo"], resolver_zona_real(c)))}
    for c in ciudades_propias
]
df_real = pd.DataFrame(resultados_reales).merge(
    df_fuzzy[["ciudad_propia", "codigo_sap", "score"]].rename(columns={"codigo_sap": "zona_fuzzy"}),
    on="ciudad_propia",
)
df_real["coincide_con_fuzzy"] = df_real["zona_real"] == df_real["zona_fuzzy"]

print(f"Resueltas con catálogo real: {df_real['zona_real'].notna().sum()} de {len(df_real)}")
print(f"Coinciden fuzzy vs. real: {df_real['coincide_con_fuzzy'].sum()} de {len(df_real)}")
print()
print(df_real.sort_values("coincide_con_fuzzy").to_string())


Resueltas con catálogo real: 21 de 24
Coinciden fuzzy vs. real: 18 de 24



          ciudad_propia         zona_real                                                                    motivo         zona_fuzzy       score  coincide_con_fuzzy
2           BOGOTA D.C.               NaN                                           no es municipio DANE reconocido         ZTC_BOGOTA   90.000000               False
19             RIONEGRO   ZTC_ORIENTE_ANT                           tier2: 'RIONEGRO' vía ZonasTarificacionRelacion  ZC_RIONEGRO (ANT)   90.000000               False
18        PUERTO NARINO               NaN  municipio DANE 'PUERTO NARINO' válido, sin zona de tarificación asociada        ZC_PTO NARE   67.500000               False
16             MOSQUERA        ZTC_BOGOTA                           tier2: 'MOSQUERA' vía ZonasTarificacionRelacion  ZC_MOSQUERA (CUN)   90.000000               False
13              LETICIA               NaN        municipio DANE 'LETICIA' válido, sin zona de tarificación asociada           ZC_UTICA   66.666667               Fals

## Paso 3c — Verificación de catálogos adicionales: ClaseContenedor y TipoCarroceria

Dos gaps más quedaban documentados como "sin catálogo disponible":
`ClaseContenedor` (H-003/L-03) y `TIPOCARROSERIA` (H-017/L-12). Ya se
consiguieron ambos catálogos
(`tblSAPClaseContenedor_202609142051.csv`,
`tblSAPTiposCarroseria_202609142052.csv`) — se verifica aquí la
traducción completa contra los valores reales que aparecen en
`lineas_solicitud`/`tarifa-plaza`/`historico-sap`.

In [6]:
lineas_solicitud = tablas["lineas_solicitud"]

# --- ClaseContenedor (H-003/L-03) ---
clase_contenedor = pd.read_csv(RUTA_CORP / "tblSAPClaseContenedor_202609142051.csv")
dic_contenedor = dict(zip(clase_contenedor["DESCRIPCION"].str.strip().str.upper(), clase_contenedor["ID"].astype(str)))

vals_propios_cnt = lineas_solicitud["claseContenedor"].dropna().str.strip().str.upper().unique()
print("--- ClaseContenedor: propio -> ID SAP -> cobertura en tarifa-plaza ---")
for v in vals_propios_cnt:
    id_sap = dic_contenedor.get(v)
    n_en_tarifa = (tarifa_plaza["clasecontenedor"] == id_sap).sum() if id_sap else 0
    print(f"{v} -> ID {id_sap} -> {n_en_tarifa} filas en tarifa-plaza")

# --- TipoCarroceria (H-017/L-12) ---
carroseria = pd.read_csv(RUTA_CORP / "tblSAPTiposCarroseria_202609142052.csv")
dic_carroseria = dict(zip(carroseria["CODIGO"].astype(str).str.strip(), carroseria["DESCRIPCION"]))

print()
print("--- TipoCarroceria en tarifa-plaza: código -> descripción real ---")
for codigo, cnt in tarifa_plaza["tipocarroseria"].value_counts(dropna=False).items():
    if pd.isna(codigo):
        print(f"(vacío) -> {cnt} filas")
        continue
    print(f"{codigo} ({cnt} filas) -> {dic_carroseria.get(str(codigo).strip(), 'SIN TRADUCCIÓN')}")

print()
print("--- TipoCarroceria en historico-sap: código -> descripción real ---")
for codigo, cnt in historico_sap["tipocarroseria"].value_counts(dropna=False).items():
    if pd.isna(codigo):
        print(f"(vacío) -> {cnt} filas")
        continue
    print(f"{codigo} ({cnt} filas) -> {dic_carroseria.get(str(codigo).strip(), 'SIN TRADUCCIÓN')}")


--- ClaseContenedor: propio -> ID SAP -> cobertura en tarifa-plaza ---
CNT20.IT -> ID 13 -> 171 filas en tarifa-plaza
CNT20.ST -> ID 8 -> 1641 filas en tarifa-plaza

--- TipoCarroceria en tarifa-plaza: código -> descripción real ---
9 (7280 filas) -> PLANCHON
2 (6897 filas) -> ESTACAS
4 (4884 filas) -> FURGON
(vacío) -> 3948 filas
1 (5 filas) -> CAMABAJA
16 (3 filas) -> VOLCO
11 (3 filas) -> PORTACONTENEDORES

--- TipoCarroceria en historico-sap: código -> descripción real ---
(vacío) -> 9746 filas
2 (5930 filas) -> ESTACAS
4 (4936 filas) -> FURGON
9 (4701 filas) -> PLANCHON
11 (4 filas) -> PORTACONTENEDORES
1 (1 filas) -> CAMABAJA


**Resultado: 100% de cobertura en ambos catálogos, sin ambigüedad.**

- `ClaseContenedor`: los 2 valores no nulos en `lineas_solicitud`
  (`CNT20.IT`, `CNT20.ST`) traducen exacto a sus ID SAP (13 y 8) y
  ambos códigos existen en volumen real en `tarifa-plaza` (171 y 1.641
  filas) — cierra H-003/L-03, ya no es "no verificable".
- `TIPOCARROSERIA`: los 6 códigos no nulos en `tarifa-plaza` y los 5 en
  `historico-sap` traducen el 100% sin ningún "SIN TRADUCCIÓN" —
  `9`=Planchón, `2`=Estacas, `4`=Furgón son los tres dominantes por
  volumen. Cierra H-017/L-12.

## Paso 4 — Cobertura geográfica general por fuente (§9 del plan)

No buscamos un cruce fino aquí — solo comparar, en términos generales,
si las 3 fuentes cubren geografías parecidas. Para SAP (tarifa-plaza,
23.020 filas reales) sí tiene sentido un "top" por frecuencia real. Para
catalogo_rutas (29 rutas) y piso-sicetac (muestra armada a propósito),
"frecuencia" no es un dato natural — ahí solo comparamos qué ciudades
aparecen, no cuántas veces.


In [7]:
print("--- Top 10 zonas más frecuentes en tarifa-plaza (origen+destino combinados) ---")
todas_zonas = pd.concat([tarifa_plaza["origen"], tarifa_plaza["destino"]])
print(todas_zonas.value_counts().head(10))
print()

print("--- Ciudades únicas en catalogo_rutas (propio, 24 en total) ---")
print(sorted(set(rutas_validas["origen_limpio"]) | set(rutas_validas["destino_limpio"])))


--- Top 10 zonas más frecuentes en tarifa-plaza (origen+destino combinados) ---
ZTC_CARTAGENA       6052
ZC_BUENAVENTURA     4656
ZTC_BARRANQUILLA    3054
ZTC_BOGOTA          2342
ZTC_VABURRA         1930
ZTC_NORTE_BOGOTA    1652
ZTC_CALI            1410
ZTC_ORIENTE_ANT     1308
ZC_SANTA MARTA      1148
ZTC_CALOTO           913
Name: count, dtype: int64

--- Ciudades únicas en catalogo_rutas (propio, 24 en total) ---
['BARRANQUILLA', 'BOGOTA', 'BOGOTA D.C.', 'BUCARAMANGA', 'BUENAVENTURA', 'CALI', 'CARTAGENA', 'CARTAGENA DE INDIAS', 'COPACABANA', 'CUCUTA', 'DOSQUEBRADAS', 'FUNZA', 'GUARNE', 'LETICIA', 'MANIZALES', 'MEDELLIN', 'MOSQUERA', 'PEREIRA', 'PUERTO NARINO', 'RIONEGRO', 'SANTA MARTA', 'SANTIAGO DE CALI', 'SOACHA', 'YUMBO']


In [8]:
piso_sicetac_nucleo = tablas["piso-sicetac_nucleo"]
piso_sicetac_raras = tablas["piso-sicetac_raras"]

texto_sicetac_completo = " | ".join(
    piso_sicetac_nucleo["nombreruta"].dropna().tolist()
    + piso_sicetac_raras["nombreruta"].dropna().tolist()
)

ciudades_propias = sorted(set(rutas_validas["origen_limpio"]) | set(rutas_validas["destino_limpio"]))

for ciudad in ciudades_propias:
    aparece = ciudad in texto_sicetac_completo
    print(f"{ciudad}: {'sí aparece' if aparece else 'NO aparece'} en SICE-TAC")


BARRANQUILLA: sí aparece en SICE-TAC
BOGOTA: sí aparece en SICE-TAC
BOGOTA D.C.: NO aparece en SICE-TAC
BUCARAMANGA: sí aparece en SICE-TAC
BUENAVENTURA: sí aparece en SICE-TAC
CALI: sí aparece en SICE-TAC
CARTAGENA: sí aparece en SICE-TAC
CARTAGENA DE INDIAS: NO aparece en SICE-TAC
COPACABANA: sí aparece en SICE-TAC
CUCUTA: sí aparece en SICE-TAC
DOSQUEBRADAS: sí aparece en SICE-TAC
FUNZA: sí aparece en SICE-TAC
GUARNE: sí aparece en SICE-TAC
LETICIA: NO aparece en SICE-TAC
MANIZALES: sí aparece en SICE-TAC
MEDELLIN: sí aparece en SICE-TAC
MOSQUERA: sí aparece en SICE-TAC
PEREIRA: sí aparece en SICE-TAC
PUERTO NARINO: NO aparece en SICE-TAC
RIONEGRO: NO aparece en SICE-TAC
SANTA MARTA: sí aparece en SICE-TAC
SANTIAGO DE CALI: NO aparece en SICE-TAC
SOACHA: NO aparece en SICE-TAC
YUMBO: sí aparece en SICE-TAC


## Síntesis del notebook 06 — Temporal y Geográfico (Fases 7-8, §8-9 del plan)

**Fase 7 — Temporal (§8):**

Primer intento (Paso 1, sobre `tarifa-plaza`) mezclaba todas las
categorías de negocio en un solo número por mes — método corregido:
se repitió (Paso 2) sobre `historico-sap` (versiones reales en el
tiempo, no una sola foto), separado por `LineaNegocio`. **No se
reproduce el salto limpio del ~17% en marzo 2026 que encontró el EDA-1**
— de hecho, las 4 categorías principales *bajan* de febrero a marzo
2026. Sí hay indicios de una subida gradual en `Nacional` entre
noviembre 2025 y febrero 2026 (+18,5%, pero repartida en 4 meses, no
concentrada). Conclusión: no se confirma el mismo patrón exacto, aunque
hay señales parciales de subida en el mismo período general — se
documenta así, sin forzar una confirmación que el dato no sostiene con
claridad.

**Fase 8 — Geográfico (§9), el trabajo fino que quedaba pendiente:**

- **Normalización con `rapidfuzz`** (Paso 3, aproximación inicial): de
  24 ciudades únicas del catálogo propio, 22 (91,7%) encuentran match
  confiable (≥90%) contra el vocabulario de zonas SAP. 2 ciudades
  (`Leticia`, `Puerto Nariño`, ambas del Amazonas) dan match de baja
  confianza (66-67%) con lugares geográficamente incorrectos —
  declaradas explícitamente "sin match confiable", no aceptadas a
  ciegas.
- **Verificación con el catálogo real (Paso 3b, post-cierre) —
  `tblSAPDANE` + `tblZonasTarificacionRelacion`, ya conseguidas.** Se
  reprodujo el mecanismo real de 2 niveles (zona directa, luego
  municipio DANE → zona vía tabla de relación). Resultado: **21 de 24
  ciudades (87,5%) resueltas con el catálogo real — pero solo 18/24
  coinciden con lo que había dado rapidfuzz.** El fuzzy matching se
  equivocaba con **confianza alta (90%)** en 3 ciudades satélite:
  `Mosquera` y `Soacha` (fuzzy: su propio código aislado
  `ZC_MOSQUERA (CUN)`/`ZC_SOACHA (CUN)`, que no corresponde a ninguna
  zona real de tarifa-plaza; real: ambas son satélites de Bogotá,
  resuelven a `ZTC_BOGOTA`) y `Rionegro` (fuzzy: `ZC_RIONEGRO (ANT)`;
  real: `ZTC_ORIENTE_ANT`, la zona metropolitana correcta). Es decir,
  fuzzy no solo aproximaba — en estos 3 casos daba una zona
  **equivocada** con la confianza suficiente para pasar desapercibida.
  `Leticia`/`Puerto Nariño` se reconfirman sin zona SAP, pero ahora con
  certeza (municipio DANE real pero sin fila en la tabla de relación ni
  en tarifa-plaza) — antes era "sin match confiable", ahora es "no
  tiene zona de tarificación, punto". `Bogotá D.C.` (la única variante
  de Bogotá con ese sufijo exacto) no resuelve por el catálogo real
  porque `tblSAPDANE` no incluye una fila con ese nombre de municipio —
  no afecta ningún análisis de este EDA porque las demás variantes
  (`Bogotá`, `Bogota`) sí resuelven directo (tier 1).
- **Verificación de 2 catálogos más (Paso 3c, post-cierre) —
  `ClaseContenedor` y `TIPOCARROSERIA`, ya conseguidos.** 100% de
  cobertura en ambos: los 2 valores de `ClaseContenedor` en
  `lineas_solicitud` traducen exacto (cierra H-003/L-03); los 6+5
  códigos de `TIPOCARROSERIA` en `tarifa-plaza`/`historico-sap`
  traducen exacto (cierra H-017/L-12).
- **Cobertura geográfica general** (Pasos 4-5): 7 de las 10 zonas SAP
  más frecuentes coinciden con ciudades del catálogo propio (los
  puertos principales + las 3 ciudades más grandes). Contra SICE-TAC,
  20 de 24 ciudades (83%) aparecen (contando variantes de nombre); las
  4 que no — 2 son casos remotos genuinos (Amazonas), 2 probablemente
  son solo limitación de la muestra chica de 151 filas, no ausencia
  real en la tabla completa.
- **Corrección de reproducibilidad (revisión post-cierre):** el
  diccionario `vocabulario_sap` (Paso 3) se armaba iterando sobre un
  `set` de Python — encontramos 14 colisiones reales (ciudades con
  tanto código `ZC_` como `ZTC_` a la vez, ej. `ZC_BOGOTA`/
  `ZTC_BOGOTA`), donde el resultado podía variar entre corridas sin un
  orden fijo. Corregido con `sorted()` para que el resultado sea
  siempre el mismo — no cambió ningún número ya reportado (mismo
  91,7% de match, mismos matches individuales verificados), solo
  garantiza que se repita igual en el futuro.

**Impacto en H-005:** ninguno de los 3 errores de fuzzy (Mosquera,
Soacha, Rionegro) ni el caso sin resolver (Bogotá D.C.) participan en
las 32 líneas `Fuente=Tarifario` usadas en H-005 (notebook 05) — esas 8
rutas únicas ya resolvían igual por ambos métodos (tier 1, match
directo). El hallazgo de H-005 no necesita recalcularse por este
descubrimiento.

**Para la extracción futura del modelo (actualiza recomendación
previa):** ya no falta pedir ninguna de las tablas identificadas por
este EDA — `tblsapdane`, `tblZonasTarificacionRelacion`,
`tblSAPClaseContenedor` y `tblSAPTiposCarroseria` se consiguieron y ya
se usaron para reemplazar aproximaciones por el mecanismo/catálogo
real.

**Qué le queda listo al siguiente notebook:** entendimiento completo de
qué tan bien conecta la geografía entre las 3 fuentes, ahora con el
mecanismo real (no aproximado) — y evidencia concreta de que la
aproximación por fuzzy matching, aunque útil como estimado, puede fallar
con alta confianza en ciudades satélite de zonas metropolitanas. Además,
2 catálogos de traducción más (ClaseContenedor, TipoCarroceria) quedan
100% verificados. Siguen las Fases 9-11 del plan (outliers, sesgos,
vacíos) antes de la síntesis final de esta HU.
